* Baseline 2: – Sentence Embedding + Cosine
- Dùng mô hình embedding phổ biến
- Bắt được ngữ nghĩa
- Chạy ổn định, có kết quả
- So sánh được với TF-IDF

In [1]:
import pandas as pd
import numpy as np
import re
import os

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
jobs = pd.read_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/it_jobs.xlsx")
resumes = pd.read_csv("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

job_texts = (
    jobs["title"].fillna("") + " " + jobs["cleaned_description"].fillna("")
).apply(clean_text)


In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [4]:
job_embeddings = model.encode(
    job_texts.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)


Batches: 100%|██████████| 344/344 [01:58<00:00,  2.92it/s]


In [5]:
def recommend_jobs_from_cv_embedding(cv_index, top_k=10):
    cv_text = clean_text(resumes.loc[cv_index, "Resume"])
    cv_embedding = model.encode(
        [cv_text],
        normalize_embeddings=True
    )

    scores = cosine_similarity(cv_embedding, job_embeddings).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = jobs.loc[top_idx, ["title", "company", "location"]].copy()
    results["score"] = scores[top_idx]
    return results


In [6]:
recommend_jobs_from_cv_embedding(cv_index=0, top_k=10)


,title,company,location,score
14341,Programmer Analyst,AI Vector,"Provo, UT, US",0.568459
10579,German NLU,Wipro,"Austin, TX, US",0.553800
18152,"Analyst, Artifical Intelligence",Walgreens,"Deerfield, IL, US",0.550770
20728,IT Sr. Data Warehousing Programmer,Health Alliance Plan of Michigan,"Troy, MI, US",0.544164
20736,IT Sr. Data Warehousing Programmer,Henry Ford Health,"Troy, MI, US",0.544164
19067,Sr. Collibra Developer,Vminds technologies Inc,"Burbank, CA, US",0.543295
8317,"Director, Engineering",NaN,"Remote, US",0.542746
21455,Support Engineer III,Link Computer Corp,"Bellwood, PA, US",0.537228
8951,"Business Intel Engineer, Santos",Amazon.com,"Seattle, WA, US",0.535428
17134,Coder/Configurator/Programmer,"QRC Group, LLC","Gurabo, PR, US",0.534269


In [7]:
os.makedirs("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_emb", exist_ok=True)

for cv_index in [0, 5, 20, 100]:
    results = recommend_jobs_from_cv_embedding(cv_index=cv_index, top_k=10)
    output_path = f"/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_emb/jobrec_embedding_cv{cv_index}_top10.csv"
    results.to_csv(output_path, index=False)
    print("Saved:", output_path)


Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_emb/jobrec_embedding_cv0_top10.csv
Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_emb/jobrec_embedding_cv5_top10.csv
Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_emb/jobrec_embedding_cv20_top10.csv
Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_emb/jobrec_embedding_cv100_top10.csv
